# Day 12 — Specialization C: Time Series
Forecast with Prophet (classical) and an LSTM (deep).
Dataset: airline passengers (small, classic, no download needed).

Install: `pip install prophet pandas numpy matplotlib torch`


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url)
df.columns = ['ds', 'y']
df['ds'] = pd.to_datetime(df['ds'])
df.plot(x='ds', y='y', title='Monthly airline passengers'); plt.show()


## 1. Prophet baseline

In [ ]:
from prophet import Prophet
split = int(len(df) * 0.85)
train, test = df.iloc[:split], df.iloc[split:]
m = Prophet(seasonality_mode='multiplicative', yearly_seasonality=True)
m.fit(train)
future = m.make_future_dataframe(periods=len(test), freq='MS')
forecast = m.predict(future)
rmse = float(np.sqrt(((forecast.tail(len(test))['yhat'].values - test['y'].values) ** 2).mean()))
print(f'Prophet RMSE = {rmse:.2f}')
m.plot(forecast); plt.show()


## 2. LSTM forecaster (PyTorch)

In [ ]:
import torch, torch.nn as nn

vals = df['y'].values.astype('float32')
mu, sd = vals[:split].mean(), vals[:split].std()
vn = (vals - mu) / sd
WINDOW = 12  # one year of history

def make_windows(arr, w):
    X, y = [], []
    for i in range(len(arr) - w):
        X.append(arr[i:i + w]); y.append(arr[i + w])
    return np.array(X), np.array(y)

X_tr, y_tr = make_windows(vn[:split], WINDOW)
X_te, y_te = make_windows(vn[split - WINDOW:], WINDOW)
X_tr = torch.tensor(X_tr).unsqueeze(-1); y_tr = torch.tensor(y_tr)
X_te = torch.tensor(X_te).unsqueeze(-1); y_te = torch.tensor(y_te)

class LSTMReg(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1]).squeeze(-1)

model = LSTMReg()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()
for epoch in range(300):
    opt.zero_grad(); loss = loss_fn(model(X_tr), y_tr); loss.backward(); opt.step()
    if epoch % 50 == 0: print(f'ep {epoch} loss {loss.item():.4f}')
with torch.no_grad():
    pred = model(X_te).numpy() * sd + mu
actual = y_te.numpy() * sd + mu
rmse_lstm = float(np.sqrt(((pred - actual) ** 2).mean()))
print(f'LSTM RMSE = {rmse_lstm:.2f}')
plt.plot(actual, label='actual'); plt.plot(pred, label='lstm'); plt.legend(); plt.title('LSTM forecast');


### Exercises
- Add exogenous regressors to Prophet (e.g. holiday effects).
- Try a multi-step forecast (predict next 12 months at once).
- Replace LSTM with a Temporal Fusion Transformer (`pip install pytorch-forecasting`).
